# UrbanSound8K - Environmental Sound Classification
End-to-end ML pipeline: acquisition → preprocessing → model → evaluation → prediction → retraining.

**Before running:** set `DATASET` below to your downloaded `UrbanSound8K` folder.

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '..')                      # so `src` is importable from /notebook
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import librosa, librosa.display
from src.config import *
from src import preprocessing as pp, model as ml, prediction, visualization
from src.data_acquisition import build_dataset

DATASET = Path('/path/to/UrbanSound8K')       # <-- EDIT THIS

## 1. Data acquisition
Folds 1-9 → `data/train`, fold 10 → `data/test`. We keep the official fold boundaries because clips cut from the same recording share a fold; reshuffling would leak near-duplicates into the test set.

In [ ]:
meta = pd.read_csv(DATASET / 'metadata' / 'UrbanSound8K.csv')
build_dataset(DATASET)                         # copies wavs into data/train/<class>/ and data/test/<class>/
meta.head()

## 2. Exploratory analysis & visualisations (the 'story' of the data)

In [ ]:
# Feature 1: class balance
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
meta['class'].value_counts().plot.bar(ax=ax[0], title='Clips per class')
meta['salience'].map({1: 'foreground', 2: 'background'}).value_counts().plot.bar(ax=ax[1], title='Salience (foreground vs background)')
plt.tight_layout(); plt.show()
meta.groupby('class').apply(lambda d: (d['end'] - d['start']).mean()).round(2).rename('mean clip length (s)')

In [ ]:
# Feature 2: spectral centroid / zero-crossing rate / RMS per class (sampled)
viz = visualization.compute_viz(sample_per_class=30, force=True)
feat_df = pd.DataFrame([{'class': c, 'feature': k, 'value': v}
                        for c, d in viz['features'].items() for k, vals in d.items() for v in vals])
g = sns.catplot(data=feat_df, x='value', y='class', col='feature', kind='box', sharex=False, height=4, aspect=1)
plt.show()

In [ ]:
# Feature 3: mean log-mel spectrogram per class
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, (c, S) in zip(axes.ravel(), viz['mean_spectrograms'].items()):
    ax.imshow(np.array(S), origin='lower', aspect='auto', cmap='magma'); ax.set_title(c); ax.axis('off')
plt.show()

### Interpretation (fill in after viewing your plots)
1. **Class balance:** car_horn and gun_shot have far fewer clips than the other classes → accuracy alone can mislead; we report macro precision/recall/F1 and use class weights.
2. **Spectral features:** *(which classes separate cleanly on centroid / ZCR / RMS? which overlap, e.g. air_conditioner vs engine_idling?)*
3. **Mean spectrograms:** *(describe the visual signature of siren, gun_shot, jackhammer... and how that explains the confusion matrix later)*

## 3. Preprocessing
Every clip is resampled to 22.05 kHz mono, padded/trimmed to 4 s, converted to a 64-band log-mel spectrogram (dB) and standardised per clip → input shape `(64, 173, 1)`.

In [ ]:
sample = next(Path(TRAIN_DIR / 'dog_bark').glob('*.wav'))
y = pp.load_audio(str(sample)); S = pp.mel_db(y, normalize=False)[..., 0]
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
librosa.display.waveshow(y, sr=SR, ax=ax[0]); ax[0].set_title('Waveform (fixed 4 s)')
im = ax[1].imshow(S, origin='lower', aspect='auto', cmap='magma'); ax[1].set_title('Log-mel spectrogram'); plt.colorbar(im, ax=ax[1])
plt.show()

In [ ]:
X, y, groups, _ = pp.load_dataset([TRAIN_DIR])
Xt, yt, _, _ = pp.load_dataset([TEST_DIR])
print('train', X.shape, 'test', Xt.shape)

## 4. Model creation
Baseline CNN vs optimised CNN (BatchNorm, Dropout, L2, global average pooling, class weights, early stopping, LR-on-plateau). The train/validation split is grouped by source recording to avoid leakage.

In [ ]:
base = ml.build_baseline()
h_base = ml.fit_model(base, X, y, groups, epochs=20)
opt = ml.build_cnn(); opt.summary()
h_opt = ml.fit_model(opt, X, y, groups, epochs=40)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, h in [('baseline', h_base), ('optimised', h_opt)]:
    ax[0].plot(h['val_accuracy'], label=f'{name} val'); ax[0].plot(h['accuracy'], '--', label=f'{name} train')
    ax[1].plot(h['val_loss'], label=f'{name} val'); ax[1].plot(h['loss'], '--', label=f'{name} train')
ax[0].set_title('Accuracy'); ax[1].set_title('Loss'); ax[0].legend(); plt.show()

## 5. Model testing - all metrics on the held-out fold 10

In [ ]:
m_base, m_opt = ml.evaluate(base, Xt, yt), ml.evaluate(opt, Xt, yt)
pd.DataFrame({'baseline': m_base, 'optimised': m_opt}).loc[['accuracy', 'loss', 'precision_macro', 'recall_macro', 'f1_macro']].round(3)

In [ ]:
print(pd.DataFrame(m_opt['report']).T.round(3))
plt.figure(figsize=(8, 6))
sns.heatmap(m_opt['confusion_matrix'], annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix - optimised CNN'); plt.show()

## 6. Save the production model
`train_model` trains the optimised CNN, evaluates it, writes `models/urbansound_cnn.h5` and records metrics in `models/registry.json`.

In [ ]:
entry = ml.train_model(epochs=40)
print('model version', entry['version'], '| accuracy', round(entry['metrics']['accuracy'], 3), '| macro-F1', round(entry['metrics']['f1_macro'], 3))

## 7. Prediction function

In [ ]:
test_file = next(Path(TEST_DIR / 'siren').glob('*.wav'))
res = prediction.predict(str(test_file))
print('true: siren | predicted:', res['class'], f"({res['confidence']:.1%})")
pd.Series(res['probabilities']).sort_values().plot.barh(title='Class probabilities'); plt.show()

## 8. Retraining
Simulates a user uploading new labelled data. In production this is done from the UI / `POST /upload` + `POST /retrain`; retraining also fires automatically once `AUTO_RETRAIN_THRESHOLD` files are waiting.

In [ ]:
import shutil
(UPLOAD_DIR / 'dog_bark').mkdir(parents=True, exist_ok=True)
for p in list((TEST_DIR / 'dog_bark').glob('*.wav'))[:20]:          # stand-in for new user data
    shutil.copy(p, UPLOAD_DIR / 'dog_bark' / f'upload_{p.name}'.replace('-', '_'))
print('pending uploads:', ml.count_uploads())
new = ml.retrain_model(epochs=5)
print('v', new['version'], '| promoted:', new['promoted'], '| accuracy', round(new['metrics']['accuracy'], 3))
pd.DataFrame([{'version': v['version'], 'kind': v['kind'], 'n_train': v['n_train'],
               'accuracy': v['metrics']['accuracy'], 'f1_macro': v['metrics']['f1_macro']}
              for v in ml.load_registry()['versions']])